<a href="https://colab.research.google.com/github/nisabulut/digital-image-processing/blob/main/10_1_goruntu_isleme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
%%bash
# OpenCV geliştirici paketlerini yükleyelim
sudo apt-get update -y && sudo apt-get install -y libopencv-dev

cat << 'EOF' > stereo.cpp
#include <opencv2/opencv.hpp>
#include <iostream>

int main() {
    cv::Mat imgLeft  = cv::imread("scene1.row3.col1.ppm", cv::IMREAD_GRAYSCALE);
    cv::Mat imgRight = cv::imread("scene1.row3.col3.ppm", cv::IMREAD_GRAYSCALE);

    if (imgLeft.empty() || imgRight.empty()) {
        std::cerr << "Goruntuler yuklenemedi!" << std::endl;
        return -1;
    }

    int minDisparity = 0;
    int numDisparities = 16 * 8;
    int blockSize = 5;
    int cn = imgLeft.channels();

    cv::Ptr<cv::StereoSGBM> sgbm = cv::StereoSGBM::create(
        minDisparity, numDisparities, blockSize,
        8 * cn * blockSize * blockSize,
        32 * cn * blockSize * blockSize,
        1, 63, 10, 100, 32,
        cv::StereoSGBM::MODE_SGBM_3WAY
    );

    cv::Mat disp16S, dispFloat, dispNormalized, dispColor;
    sgbm->compute(imgLeft, imgRight, disp16S);
    disp16S.convertTo(dispFloat, CV_32F, 1.0 / 16.0);

    double minVal, maxVal;
    cv::minMaxLoc(dispFloat, &minVal, &maxVal);
    double scale = (maxVal - minVal > 0) ? (255.0 / (maxVal - minVal)) : 1.0;
    dispFloat.convertTo(dispNormalized, CV_8U, scale, -minVal * scale);

    cv::applyColorMap(dispNormalized, dispColor, cv::COLORMAP_TURBO);
    cv::imwrite("disparity_color.png", dispColor);

    std::cout << "Tamamlandi: disparity_color.png olusturuldu." << std::endl;
    return 0;
}
EOF

g++ stereo.cpp -o stereo `pkg-config --cflags --libs opencv4`
./stereo

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Fetched 3,917 B in 31s (125 B/s)
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
libopencv-dev is already the newest version (4.6.0+dfsg-13.1ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 68 not upgraded.
Tamamlandi: disparity_color.png olusturuldu.


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
